In [2]:
# 重启后先运行这个 cell 重新加载环境模块
import importlib
import sys
if 'environment' in sys.modules:
    del sys.modules['environment']
import environment
from environment import WumpusWorldEnvironment
importlib.reload(environment)
from environment import WumpusWorldEnvironment

import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque
import matplotlib.pyplot as plt

# 检查 GPU 是否可用
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# grid environment
from environment import WumpusWorldEnvironment

env_grid = WumpusWorldEnvironment(observation_type='vector', action_type='discrete')

obs_sample, _ = env_grid.reset()
#define the input dimension

if isinstance(obs_sample, (int, np.integer)):
    grid_obs_dim = 1 # 或者根据你的地图大小设为格子总数（如果要用One-hot）
else:
    grid_obs_dim = len(obs_sample)

grid_action_dim = env_grid.action_space.n

In [4]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(QNetwork, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim) # 线性输出，不加 Softmax 或 Sigmoid
        )

    def forward(self, x):
        return self.fc(x)

In [5]:
# Prioritized Replay Buffer

class PrioritizedReplayBuffer:
    def __init__(self, capacity, alpha=0.6):
        self.capacity = capacity
        self.alpha = alpha  # 决定优先级的程度 (0=均匀分布, 1=完全优先级)
        self.buffer = []
        self.pos = 0
        self.priorities = np.zeros((capacity,), dtype=np.float32)
    
    def push(self, state, action, reward, next_state, done):
        # 新样本赋予当前最大优先级，确保它们至少被训练一次
        max_prio = self.priorities.max() if self.buffer else 1.0
        
        if len(self.buffer) < self.capacity:
            self.buffer.append((state, action, reward, next_state, done))
        else:
            self.buffer[self.pos] = (state, action, reward, next_state, done)
        
        self.priorities[self.pos] = max_prio
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size, beta=0.4):
        """
        beta: 采样权重的修正系数 (随着训练增加，逐步从 0.4 增加到 1.0)
        """
        if len(self.buffer) == self.capacity:
            prios = self.priorities
        else:
            prios = self.priorities[:self.pos]
            
        # 根据优先级计算采样概率 P(i) = p^a / sum(p^a)
        probs = prios ** self.alpha
        probs /= probs.sum()

        # 按概率抽取索引
        indices = np.random.choice(len(self.buffer), batch_size, p=probs)
        samples = [self.buffer[idx] for idx in indices]

        # 计算重要性采样权重 (IS weights) 来修正偏差
        # weights = (1/N * 1/P(i))^beta
        total = len(self.buffer)
        weights = (total * probs[indices]) ** (-beta)
        weights /= weights.max()  # 归一化以保持训练稳定
        
        # 解压数据
        states, actions, rewards, next_states, dones = zip(*samples)
        
        return (np.array(states, dtype=np.float32), 
                np.array(actions, dtype=np.int64), 
                np.array(rewards, dtype=np.float32), 
                np.array(next_states, dtype=np.float32), 
                np.array(dones, dtype=np.float32), 
                indices, 
                np.array(weights, dtype=np.float32))

    def update_priorities(self, batch_indices, batch_priorities):
        for idx, prio in zip(batch_indices, batch_priorities):
            # 加上 1e-6 的 epsilon 确保每个样本至少有微小的被选中概率
            self.priorities[idx] = prio + 1e-6

    def __len__(self):
        return len(self.buffer)

In [6]:
# Prioritized Experience Replay DQN Agent
class PER_DQNAgent:
    def __init__(self, state_dim, action_dim, device="cpu"):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.device = device
        
        # 1. 网络定义
        self.policy_net = QNetwork(state_dim, action_dim).to(device)
        self.target_net = QNetwork(state_dim, action_dim).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        
        # 2. 优化器与 PER Buffer
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=1e-3)
        self.memory = PrioritizedReplayBuffer(capacity=10000)
        
        # 3. 超参数
        self.gamma = 0.99
        self.epsilon = 1.0
        self.epsilon_decay = 0.996
        self.epsilon_min = 0.01
        self.beta = 0.4  # IS weight 系数
        self.beta_increment = 0.001

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.action_dim - 1)
        
        state_t = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        with torch.no_grad():
            q_values = self.policy_net(state_t)
            return q_values.argmax().item()

    def update(self, batch_size):
        if len(self.memory) < batch_size:
            return None
        
        # 采样 (返回带索引和权重的数据)
        states, actions, rewards, next_states, dones, indices, weights = \
            self.memory.sample(batch_size, beta=self.beta)
        
        # 转换为 Tensor
        states = torch.FloatTensor(states).to(self.device)
        actions = torch.LongTensor(actions).unsqueeze(1).to(self.device)
        rewards = torch.FloatTensor(rewards).to(self.device)
        next_states = torch.FloatTensor(next_states).to(self.device)
        dones = torch.FloatTensor(dones).to(self.device)
        weights = torch.FloatTensor(weights).to(self.device)

        # 1. 计算当前的 Q 值
        current_q = self.policy_net(states).gather(1, actions)

        # 2. 计算目标 Q 值 (贝尔曼方程)
        with torch.no_grad():
            next_q = self.target_net(next_states).max(1)[0]
            target_q = rewards + (1 - dones) * self.gamma * next_q

        # 3. 计算 TD-error 并更新优先级
        # 计算 Loss 之前，必须获取每个样本的误差来更新 Buffer
        td_errors = torch.abs(target_q - current_q.squeeze()).detach()
        self.memory.update_priorities(indices, td_errors.cpu().numpy())

        # 4. 计算加权 MSE Loss (PER 的核心)
        # Loss = mean(weights * (Q_target - Q_curr)^2)
        loss = (weights * (current_q.squeeze() - target_q).pow(2)).mean()

        # 5. 反向传播
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        # 6. 更新衰减系数
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
        self.beta = min(1.0, self.beta + self.beta_increment) # Beta 逐渐增加到 1.0
        
        return loss.item()

print("PER-DQN Agent is ready for training.")

PER-DQN Agent is ready for training.


In [7]:
#training loop
def train_per_dqn(env, agent, num_episodes, batch_size, target_update_freq):
    rewards_history = []
    losses_history = []
    
    for ep in range(num_episodes):
        state, _ = env.reset()
        ep_reward = 0
        done = False
        
        while not done:
            action = agent.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            # 存入 PER Buffer
            agent.memory.push(state, action, reward, next_state, done)
            
            # 更新网络
            loss = agent.update(batch_size)
            if loss:
                losses_history.append(loss)
                
            state = next_state
            ep_reward += reward
        
        # 定期同步 Target Network
        if ep % target_update_freq == 0:
            agent.target_net.load_state_dict(agent.policy_net.state_dict())
            
        rewards_history.append(ep_reward)
        
        if ep % 20 == 0:
            print(f"Episode {ep} | Reward: {ep_reward:.1f} | Epsilon: {agent.epsilon:.2f} | Beta: {agent.beta:.2f}")
            
    return rewards_history, losses_history

In [8]:
#Grid-world training
# 初始化环境
env_grid = WumpusWorldEnvironment(observation_type='vector', action_type='discrete')
agent_grid = PER_DQNAgent(state_dim=2, action_dim=4, device=device)

# 训练
print("Start training Grid-world...")
grid_rewards, grid_losses = train_per_dqn(env_grid, agent_grid, num_episodes=200, batch_size=32, target_update_freq=10)


Start training Grid-world...
Episode 0 | Reward: -104.0 | Epsilon: 1.00 | Beta: 0.40
Episode 20 | Reward: -103.0 | Epsilon: 0.16 | Beta: 0.86
Episode 40 | Reward: -101.0 | Epsilon: 0.07 | Beta: 1.00
Episode 60 | Reward: -101.0 | Epsilon: 0.06 | Beta: 1.00
Episode 80 | Reward: -105.0 | Epsilon: 0.04 | Beta: 1.00
Episode 100 | Reward: -172.0 | Epsilon: 0.01 | Beta: 1.00
Episode 120 | Reward: -101.0 | Epsilon: 0.01 | Beta: 1.00
Episode 140 | Reward: -682.0 | Epsilon: 0.01 | Beta: 1.00
Episode 160 | Reward: -463.0 | Epsilon: 0.01 | Beta: 1.00
Episode 180 | Reward: -105.0 | Epsilon: 0.01 | Beta: 1.00
